# 9.5. Recurrent Neural Network Implementation from Scratch
D2L의 Recurrent Neural Network Implementation from Scratch장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import math
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

torch.manual_seed(42)

print("PyTorch version:", torch.__version__)

## 1. RNN 직접 구현

이번 장에서는 PyTorch의 nn.RNN을 사용하지 않고 RNN을 직접 구현한다. 목표는 문자 단위(Character-level) 언어 모델을 만드는 것이다.

예를 들어 모델이 time mach까지 입력받았다면 다음 문자가 무엇일지 예측한다.

전체 구조는 다음과 같다.

```text
문자
↓
One-Hot Encoding
↓
RNN
↓
Hidden State
↓
Fully Connected Layer
↓
다음 문자 예측
```

RNN에서 가장 중요한 것은 현재 입력뿐 아니라 이전 시점의 hidden state도 함께 사용한다는 것이다.

$$
H_t = \tanh(X_tW_{xh} + H_{t-1}W_{hh} + b_h)
$$

## 2. 파라미터

RNN 내부 계산을 직접 만들기 위해 다음 파라미터들을 직접 정의할 것이다.

`W_xh`: 입력 → hidden  
`W_hh`: 이전 hidden → 현재 hidden  
`b_h` : hidden bias

## 3. RNN 파라미터 만들기

In [ ]:
class RNNScratch(d2l.Module):
    def __init__(self, num_inputs, num_hiddens, sigma=0.01):
        super().__init__()

        self.num_inputs = num_inputs
        self.num_hiddens = num_hiddens
        self.sigma = sigma

        self.W_xh = nn.Parameter(
            torch.randn(num_inputs, num_hiddens) * sigma
        )

        self.W_hh = nn.Parameter(
            torch.randn(num_hiddens, num_hiddens) * sigma
        )

        self.b_h = nn.Parameter(
            torch.zeros(num_hiddens)
        )

shape를 보면 RNN 구조를 이해하기 쉽다. 현재 입력 X_t는 W_xh와 곱해서 hidden state 크기로 바꾼다.

    X_t (batch_size, num_inputs) @ W_xh (num_inputs, num_hiddens) -> (batch_size, num_hiddens)

이전 hidden state H_(t-1)도 W_hh와 곱한다.

    H_(t-1) (batch_size, num_hiddens) @ W_hh (num_hiddens, num_hiddens) -> (batch_size, num_hiddens)

두 결과의 크기가 같기 때문에 더할 수 있다.

    현재 입력 정보 + 이전 시점의 기억

RNN은 현재 입력과 이전 hidden state를 함께 사용해 새로운 hidden state를 만든다.

## 4. RNN 순전파

RNN은 시간 순서대로 입력을 하나씩 처리한다.

In [ ]:
@d2l.add_to_class(RNNScratch)
def forward(self, inputs, state=None):

    if state is None:
        state = torch.zeros(
            (inputs.shape[1], self.num_hiddens),
            device=inputs.device
        )
    else:
        state, = state

    outputs = []

    for X in inputs:

        state = torch.tanh(
            X @ self.W_xh
            + state @ self.W_hh
            + self.b_h
        )

        outputs.append(state)

    return outputs, state

입력 shape = (num_steps, batch_size, num_inputs)

예를 들어 X.shape = (100, 2, 16)이면 for X in inputs를 실행하면 한 번 X는 (2, 16)이다.

한 time step씩 처리한다. 

$$
H_t = \tanh(X_tW_{xh} + H_{t-1}W_{hh} + b_h)
$$

```python
state = torch.tanh(
    X @ self.W_xh
    + state @ self.W_hh
    + self.b_h
)
```

코드로는 여기 부분이다. 현재 X와 이전 기억 state를 합쳐서 새로운 기억을 만든다.

## 5. RNN 출력 Shape 확인

In [ ]:
batch_size = 2
num_inputs = 16
num_hiddens = 32
num_steps = 100

rnn = RNNScratch(
    num_inputs=num_inputs,
    num_hiddens=num_hiddens
)

X = torch.ones(
    (num_steps, batch_size, num_inputs)
)

outputs, state = rnn(X)

print(len(outputs))
print(outputs[0].shape)
print(state.shape)

100개의 time step 각각에 대해 hidden state가 하나씩 만들어진다.

```text
X1 → H1
X2 + H1 → H2
X3 + H2 → H3
...
X100 + H99 → H100
```

마지막 state는 H100이다.

## 6. RNN을 언어 모델로 만들기

지금까지의 RNN은 hidden state만 만든다. 하지만 언어 모델은 다음 문자가 무엇인지 예측해야 한다.

    Hidden State -> Fully Connected Layer -> Vocabulary 크기 출력

구조가 추가되어야 한다.

$$
O_t = H_tW_{hq} + b_q
$$

여기서 출력 차원은 vocabulary 크기이다. 예를 들어서 사용 가능한 문자가 28개라면 이렇다.

    hidden state(batch, 32) -> output(batch, 28)

In [ ]:
class RNNLMScratch(d2l.Classifier):

    def __init__(self, rnn, vocab_size, lr=0.01):
        super().__init__()

        self.rnn = rnn
        self.vocab_size = vocab_size
        self.lr = lr

        self.W_hq = nn.Parameter(
            torch.randn(
                rnn.num_hiddens,
                vocab_size
            ) * rnn.sigma
        )

        self.b_q = nn.Parameter(
            torch.zeros(vocab_size)
        )

## 7. One-Hot Encoding

문자는 일반적으로 정수 index로 저장된다. 예를 들어서

```text
a -> 0
b -> 1
c -> 2
d -> 3
e -> 4
```

라고 하자. 그렇다고 c = 2라는 숫자를 RNN에 그대로 넣으면 문제가 있다. 숫자에서는 2랑 3은 가까운 관계가 있지만 c, d가 의미적으로 비슷하다는 뜻은 아니다. 그래서 범주형 데이터인 문자를 one-hot vector로 표현한다.

In [ ]:
F.one_hot(
  torch.tensor([0, 2]), 
  num_classes=5 
)

Vocabulary 크기가 $V$라면 각 문자는 길이 $V$인 vector가 된다. 언어 모델의 원래 입력은 `(batch_size, num_steps)`

이지만 one-hot encoding 후에는 `(batch_size, num_steps, vocab_size)`이렇게 된다.

RNN에서 time step부터 반복하기 쉽게 transpose하면 `(num_steps, batch_size, vocab_size)`로 만든다.

## 8. Hidden State를 문자 예측으로 변환

RNN의 각 hidden state를 vocabulary 크기의 출력으로 변환한다.

In [ ]:
@d2l.add_to_class(RNNLMScratch)
def output_layer(self, rnn_outputs):

    outputs = [
        H @ self.W_hq + self.b_q
        for H in rnn_outputs
    ]

    return torch.stack(outputs, dim=1)

@d2l.add_to_class(RNNLMScratch) # 전체 forward는 다음과 같다.
def forward(self, X, state=None):

    X = self.one_hot(X)

    rnn_outputs, state = self.rnn(
        X,
        state
    )

    return self.output_layer(rnn_outputs)

전체 흐름은 이런식이다.

```text
문자 index X
(batch, time)

↓

One-Hot
(time, batch, vocab)

↓

RNN
H1, H2, H3, ...
각각 (batch, hidden)

↓

Fully Connected(batch, time, vocab)

↓

다음 문자 확률 예측
```

## 9. Gradient Explosion과 Gradient Clipping

RNN에서는 같은 계산이 time step마다 반복된다.

    H1 -> H2 -> H3 -> H4 -> ... -> HT

따라서 역전파도 이 시간 방향을 거꾸로 지나간다. 이것을 BPTT(Backpropagation Through Time)이라고 한다.

sequence가 길어지면 gradient 계산에서 행렬 곱이 계속 반복되기 때문에 두 가지 문제가 발생할 수 있다.

`Gradient Vanishing`: gradient가 점점 0에 가까워짐  
`Gradient Explosion`: gradient가 지나치게 커짐  

특히 gradient explosion이 발생하면 한 번의 업데이트가 지나치게 커져 학습이 망가질 수 있다. 이를 막기 위해서 Gradient Clipping을 사용한다.

gradient를 $\mathbf{g}$, 최대 허용 norm을 $\theta$라고 하면

$$
\mathbf{g}
\leftarrow
\min
\left(
1,
\frac{\theta}{|\mathbf{g}|}
\right)
\mathbf{g}
$$

예를 들어서 

`gradient norm` = 10, `clip` = 1 이면 이렇게 된다.

$$
\frac{1}{10}g
$$

방향은 그대로 유지하고 크기만 줄이는 식이다. Gradient clipping은 exploding gradient를 완화하지만 vanishing gradient를 해결하는 방법은 아니다.

In [ ]:
def clip_gradients(model, max_norm):

    params = [
        p for p in model.parameters()
        if p.requires_grad and p.grad is not None
    ]

    norm = torch.sqrt(
        sum(
            torch.sum(p.grad ** 2)
            for p in params
        )
    )

    if norm > max_norm:

        scale = max_norm / norm

        for p in params:
            p.grad.mul_(scale)

## 10. RNN 언어 모델 학습

이제 The Time Machine 텍스트를 이용해 문자 단위 언어 모델을 학습한다.

In [ ]:
@d2l.add_to_class(RNNLMScratch)
def one_hot(self, X):
    # (batch_size, num_steps)
    # -> (num_steps, batch_size, vocab_size)
    return F.one_hot(
        X.T,
        self.vocab_size
    ).type(torch.float32)


@d2l.add_to_class(RNNLMScratch)
def output_layer(self, rnn_outputs):
    outputs = [
        H @ self.W_hq + self.b_q
        for H in rnn_outputs
    ]

    return torch.stack(outputs, dim=1)


@d2l.add_to_class(RNNLMScratch)
def forward(self, X, state=None):
    X = self.one_hot(X)

    rnn_outputs, state = self.rnn(
        X,
        state
    )

    return self.output_layer(rnn_outputs)

In [ ]:
data = d2l.TimeMachine(
    batch_size=1024,
    num_steps=32
)

rnn = RNNScratch(
    num_inputs=len(data.vocab),
    num_hiddens=32
)

model = RNNLMScratch(
    rnn,
    vocab_size=len(data.vocab),
    lr=1
)

trainer = d2l.Trainer(
    max_epochs=100,
    gradient_clip_val=1,
    num_gpus=1
)

trainer.fit(model, data)

num_steps=32이라 한번에 32개의 문자 sequence를 RNN이 처리한다. 그리고 num_hiddens=32는 hidden state의 크기이다.

학습 순서는 이렇다.
```text
문자 입력
↓
One-Hot
↓
RNN Forward
↓
다음 문자 예측
↓
Cross Entropy Loss
↓
BPTT
↓
Gradient 계산
↓
Gradient Clipping
↓
Parameter Update
```

## 11. 학습된 RNN으로 문자 생성하기

언어 모델은 다음 문자를 하나 예측하고, 그 결과를 다시 입력으로 넣어 계속 문장을 생성할 수 있다.

예를 들어 `입력: "it has"`이라면 이렇게 진행된다.

```text
"it has"
↓
다음 문자 예측
↓
예측된 문자 다시 입력
↓
다음 문자 예측
↓
...
```

처음 사용자가 제공한 prefix를 모델에 입력하는 구간을 warm-up이라고 한다.

```text
"i" → hidden state 갱신
"t" → hidden state 갱신
" " → hidden state 갱신
"h" → hidden state 갱신
"a" → hidden state 갱신
"s" → hidden state 갱신
```

이 과정에서는 모델의 예측값을 사용하지 않고 실제 prefix를 계속 입력한다. prefix를 모두 읽은 다음부터 모델 자신의 예측 결과를 다음 입력으로 사용한다.

간단히 구현하면 다음과 같은 구조이다.

In [ ]:
def generate(model, prefix, num_preds, vocab, device):

    state = None

    outputs = [vocab[prefix[0]]]

    for i in range(
        len(prefix) + num_preds - 1
    ):

        X = torch.tensor(
            [[outputs[-1]]],
            device=device
        )

        X = model.one_hot(X)

        rnn_outputs, state = model.rnn(
            X,
            state
        )

        # prefix를 읽는 구간
        if i < len(prefix) - 1:

            outputs.append(
                vocab[prefix[i + 1]]
            )

        # 실제 생성 구간
        else:

            Y = model.output_layer(
                rnn_outputs
            )

            next_token = int(
                Y.argmax(dim=2).reshape(1)
            )

            outputs.append(next_token)

    return ''.join(
        vocab.idx_to_token[i]
        for i in outputs
    )

In [ ]:
device = next(model.parameters()).device

text = generate(
    model=model,
    prefix="time traveller ",
    num_preds=50,
    vocab=data.vocab,
    device=device
)

print(text)

RNN이 "time traveller"라는 prefix를 hidden state에 누적하면서 다음 문자를 계속 예측할 수는 있다. 하지만 작은 vanilla RNN + greedy decoding에서는 자주 등장하는 the 같은 패턴에 빠져 반복되는 것 같다.

    이전 문자 정보 + 현재 문자 -> 다음 문자 예측

## 12. 오늘의 정리

- RNN은 현재 입력 $X_t$와 이전 hidden state $H_{t-1}$를 함께 사용한다.
- W_xh는 현재 입력 정보를 처리한다.
- W_hh는 이전 시점의 정보를 현재 시점으로 전달한다.
- hidden state는 RNN의 일종의 기억이라고 생각하면 된다.
- 문자 같은 범주형 데이터는 숫자 index 자체보다 One-Hot Encoding으로 표현할 수 있다.
- RNN hidden state를 Fully Connected Layer에 넣어 vocabulary 크기의 출력으로 변환한다.
- 각 출력은 다음 문자에 대한 점수가 된다.
- RNN의 역전파는 시간 방향으로도 진행되며 이를 BPTT라고 한다.
- sequence가 길면 gradient vanishing과 gradient explosion이 발생할 수 있다.
- Gradient Clipping은 gradient가 지나치게 커지는 것을 막는다.
- 학습된 언어 모델은 prefix를 입력받아 이후 문자를 반복적으로 생성할 수 있다.